# 📊 Análisis Exploratorio de Datos (EDA) – <Nombre del Dataset>

© 2025 Magalí Cazella Méndez. Todos los derechos reservados.

**Autora**: Magalí Cazella Méndez  
**Rol**: Analista de Datos | Científica de Datos  
**Contacto**: [LinkedIn](https://www.linkedin.com/in/magali_cazella_mendez)  
**Licencia**: [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

> Esta notebook forma parte de una estructura modular reutilizable. Su objetivo es realizar un diagnóstico exploratorio de la base de datos proporcionada, evaluando su calidad, estructura, variables clave y primeros hallazgos.


## 📦 Importación de librerías

A continuación se importan las librerías necesarias para la exploración de datos:

- `pandas`, `numpy`: manipulación y análisis de datos estructurados.
- `matplotlib`, `seaborn`: generación de visualizaciones informativas.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
pd.set_option('display.max_columns', None)


## 📂 Carga dinámica del dataset

Se carga el dataset de forma automática según su extensión.  
El archivo puede estar en formato `.csv`, `.xlsx` o `.json`.

> Recordá modificar la variable `ruta_archivo` según corresponda.


In [ ]:
# Ruta editable
ruta_archivo = 'datasets/<nombre_archivo>.csv'

if ruta_archivo.endswith('.csv'):
    df = pd.read_csv(ruta_archivo)
elif ruta_archivo.endswith('.xlsx'):
    df = pd.read_excel(ruta_archivo)
elif ruta_archivo.endswith('.json'):
    df = pd.read_json(ruta_archivo)
else:
    raise ValueError("Formato no soportado. Usá CSV, XLSX o JSON.")

df.head()


In [ ]:
## 🧾 Información general del dataset

Se muestra la cantidad de registros, columnas y los tipos de datos de cada variable.


In [ ]:
print("Dimensiones del dataset:", df.shape)
print("\nTipos de datos:\n")
print(df.dtypes)


## 🔍 Análisis de valores nulos y duplicados

Detectar valores faltantes o filas duplicadas es clave para evaluar la calidad de los datos.


In [ ]:
# Porcentaje de nulos
nulos = df.isnull().mean().sort_values(ascending=False) * 100
print("Porcentaje de valores nulos:\n", nulos[nulos > 0])

# Duplicados
print("\nRegistros duplicados:", df.duplicated().sum())


## 📈 Estadísticas descriptivas de variables numéricas

Se exploran medidas como media, desviación estándar, cuartiles y valores extremos.


In [ ]:
df.describe().T


## 🧮 Distribución de variables numéricas

Histograma por variable para entender la forma de su distribución (asimetría, sesgo, dispersión).


In [ ]:
df.select_dtypes(include='number').hist(bins=30, figsize=(15,10))
plt.suptitle('Distribución de variables numéricas')
plt.tight_layout()
plt.show()


## ⚠️ Detección visual de valores atípicos (Outliers)

Se utilizan boxplots para visualizar posibles valores extremos que podrían afectar análisis posteriores.


In [ ]:
for col in df.select_dtypes(include='number').columns:
    plt.figure(figsize=(10, 1.5))
    sns.boxplot(x=df[col])
    plt.title(f'Distribución de {col}')
    plt.show()


## 🧾 Análisis de variables categóricas

Se visualiza la distribución de frecuencias para variables tipo `object`, que representan categorías o etiquetas.


In [ ]:
cat_cols = df.select_dtypes(include='object').columns

for col in cat_cols:
    plt.figure(figsize=(8, 4))
    df[col].value_counts().plot(kind='bar')
    plt.title(f'Frecuencia de {col}')
    plt.ylabel('Cantidad')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## 🔗 Matriz de correlación

Se calcula y visualiza la correlación entre variables numéricas para detectar relaciones lineales potenciales.


In [ ]:
# Filtrar columnas estrictamente numéricas y convertir a float
num_df = df.select_dtypes(include='number').copy()

# Eliminar columnas que no sean completamente numéricas (por seguridad)
num_df = num_df.apply(pd.to_numeric, errors='coerce')

# Eliminar columnas con todos valores nulos (que no servirían en la correlación)
num_df = num_df.dropna(axis=1, how='all')

# Verificar que haya al menos 2 columnas válidas
if num_df.shape[1] > 1:
    plt.figure(figsize=(12, 8))
    corr = num_df.corr()
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", square=True)
    plt.title('Matriz de correlación')
    plt.show()
else:
    print("No hay suficientes variables numéricas válidas para calcular la correlación.")



## ✅ Observaciones del Análisis Exploratorio

- Variables con alta cantidad de nulos: …
- Columnas irrelevantes o constantes: …
- Posibles errores de tipeo o codificación: …
- Outliers notables: …
- Variables altamente correlacionadas: …

> Estas observaciones serán la base para el diseño de la Fase 2 (limpieza y transformación).


## 📌 Conclusiones Automáticas del Análisis Exploratorio

A continuación se generan automáticamente las observaciones clave del EDA, incluyendo nulos, columnas irrelevantes, posibles errores de codificación, outliers y correlaciones destacadas.


In [ ]:
from collections import defaultdict

# Diccionario para observaciones
observaciones = defaultdict(list)

# 1. Variables con nulos significativos (>30%)
nulos_pct = df.isnull().mean() * 100
for col, pct in nulos_pct.items():
    if pct > 30:
        observaciones["nulos"].append(f"{col}: {pct:.1f}%")

# 2. Columnas constantes (un único valor)
for col in df.columns:
    if df[col].nunique() <= 1:
        observaciones["constantes"].append(col)

# 3. Posibles errores de codificación (strings mal capitalizados o con espacios)
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    valores = df[col].dropna().unique()
    for val in valores:
        if isinstance(val, str) and val != val.strip():
            observaciones["errores_codificacion"].append(f"{col}: '{val}' contiene espacios")
        if isinstance(val, str) and val.lower() != val:
            observaciones["errores_codificacion"].append(f"{col}: '{val}' con mayúsculas irregulares")

# 4. Outliers con método IQR
numeric_cols = df.select_dtypes(include='number').columns
for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    outliers = df[(df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)]
    if len(outliers) > 0:
        observaciones["outliers"].append(f"{col}: {len(outliers)} valores fuera del rango IQR")

# 5. Correlaciones fuertes (> 0.7 o < -0.7)
corr = df.select_dtypes(include='number').corr()
corr_fuerte = []
for col in corr.columns:
    for other_col in corr.columns:
        if col != other_col:
            valor = corr.loc[col, other_col]
            if abs(valor) > 0.7:
                par = sorted([col, other_col])
                if tuple(par) not in corr_fuerte:
                    corr_fuerte.append(tuple(par))
                    observaciones["correlaciones"].append(f"{par[0]} vs {par[1]}: {valor:.2f}")

In [ ]:
print("## ✅ Observaciones del Análisis Exploratorio\n")

if observaciones["nulos"]:
    print("- Variables con alta cantidad de nulos:")
    for item in observaciones["nulos"]:
        print(f"  - {item}")
else:
    print("- No se encontraron columnas con nulos significativos (>30%).")

print("\n- Columnas irrelevantes o constantes:")
if observaciones["constantes"]:
    for item in observaciones["constantes"]:
        print(f"  - {item}")
else:
    print("  - Ninguna detectada.")

print("\n- Posibles errores de tipeo o codificación:")
if observaciones["errores_codificacion"]:
    for item in set(observaciones["errores_codificacion"]):
        print(f"  - {item}")
else:
    print("  - Ninguno detectado.")

print("\n- Outliers notables:")
if observaciones["outliers"]:
    for item in observaciones["outliers"]:
        print(f"  - {item}")
else:
    print("  - No se detectaron outliers mediante IQR.")

print("\n- Variables altamente correlacionadas:")
if observaciones["correlaciones"]:
    for item in observaciones["correlaciones"]:
        print(f"  - {item}")
else:
    print("  - No se encontraron correlaciones >0.7 o < -0.7.")
